In [2]:
import pandas as pd
import random
import pickle 
import string
from nltk.corpus import stopwords, wordnet
from nltk.tag import pos_tag
from nltk.stem import WordNetLemmatizer
from nltk.probability import FreqDist
from nltk.classify import NaiveBayesClassifier, accuracy
from nltk.tokenize import word_tokenize

In [5]:
DATA_PATH = "../datasets/nlp/financial_dataset.csv"
data = pd.read_csv(DATA_PATH)
data.dropna()
data.head()

,Statement,Sentiment
0,The GeoSolutions technology will leverage Bene...,positive
1,"$ESI on lows, down $1.50 to $2.50 BK a real po...",negative
2,"For the last quarter of 2010 , Componenta 's n...",positive
3,$SPY wouldn't be surprised to see a green close,positive
4,Shell's $70 Billion BG Deal Meets Shareholder ...,negative


In [6]:
# Gathering the important column
statement_list = data["Statement"]
sentiment_list = data["Sentiment"]

In [7]:
stopwords_list = stopwords.words("english")
punctuation_list = string.punctuation
lemmatizer = WordNetLemmatizer()

def get_new_tag(tag):
    if tag.startswith("J"):
        return "a"
    elif tag.startswith("V"):
        return "v"
    elif tag.startswith("N"):
        return "n"
    elif tag.startswith("R"):
        return "r"
    else:
        return "n" 
    
def preprocessing(sentence):
    sentence = sentence.lower()
    word_list = word_tokenize(sentence)

    # Remove Stopwords
    word_list = [word for word in word_list if word not in stopwords_list]

    # Remove Punc
    word_list = [word for word in word_list if word not in punctuation_list]

    # Remove Numbers
    word_list = [word for word in word_list if word.isalpha()]

    # Lemmatize
    tagging = pos_tag(word_list)
    word_list = [lemmatizer.lemmatize(text, get_new_tag(tag)) for text, tag in tagging]

    return word_list

In [8]:
# Joining all the Statement List
all_statement = (" ").join(statement_list)
all_statement = preprocessing(all_statement)

print(all_statement)

['geosolutions', 'technology', 'leverage', 'benefon', 'gps', 'solution', 'provide', 'location', 'base', 'search', 'technology', 'community', 'platform', 'location', 'relevant', 'multimedia', 'content', 'new', 'powerful', 'commercial', 'model', 'esi', 'low', 'bk', 'real', 'possibility', 'last', 'quarter', 'componenta', 'net', 'sale', 'double', 'period', 'year', 'earlier', 'move', 'zero', 'profit', 'loss', 'spy', 'would', 'surprise', 'see', 'green', 'close', 'shell', 'billion', 'bg', 'deal', 'meet', 'shareholder', 'skepticism', 'ssh', 'communication', 'security', 'corp', 'stock', 'exchange', 'release', 'october', 'pm', 'company', 'update', 'full', 'year', 'outlook', 'estimate', 'result', 'remain', 'loss', 'full', 'year', 'kone', 'net', 'sale', 'rise', 'first', 'nine', 'month', 'circulation', 'revenue', 'increase', 'finland', 'sweden', 'sap', 'disappoints', 'software', 'license', 'real', 'problem', 'cloud', 'growth', 'trail', 'msft', 'orcl', 'goog', 'crm', 'adbe', 'http', 'subdivision', '

In [9]:
# Freq_dist Sentences
freq_dist = FreqDist(all_statement)
word_features = [word for word, freq in freq_dist.most_common(500)]
print(word_features)

['eur', 'mn', 'profit', 'sale', 'company', 'net', 'say', 'finnish', 'million', 'year', 'period', 'http', 'quarter', 'mln', 'share', 'increase', 'loss', 'compare', 'group', 'first', 'rise', 'oyj', 'market', 'today', 'operating', 'total', 'euro', 'new', 'price', 'finland', 'operate', 'service', 'percent', 'buy', 'high', 'aapl', 'business', 'report', 'also', 'contract', 'result', 'long', 'order', 'stock', 'per', 'expect', 'correspond', 'decrease', 'operation', 'hel', 'last', 'go', 'short', 'growth', 'earnings', 'solution', 'tsla', 'earlier', 'look', 'pct', 'bank', 'third', 'grow', 'technology', 'low', 'continue', 'helsinki', 'sell', 'product', 'agreement', 'system', 'cost', 'month', 'sign', 'estimate', 'maker', 'second', 'see', 'close', 'billion', 'https', 'revenue', 'make', 'good', 'fell', 'strong', 'time', 'customer', 'plc', 'nokia', 'usd', 'financial', 'half', 'deal', 'day', 'eps', 'end', 'get', 'corporation', 'u', 'line', 'omx', 'well', 'corresponding', 'support', 'back', 'move', 'spy

In [10]:
# Extract Features
def extract_features(sentence):
    features = {}

    for word in word_features:
        features[word] = ((word in sentence))

    return features

In [11]:
def get_features_sets():
    labeled_data = zip(statement_list, sentiment_list)
    features_sets = []

    for statement, sentiment in labeled_data:
        sentence = preprocessing(statement)
        features = extract_features(sentence)
        features_sets.append((features,sentiment))

    return features_sets

In [15]:
# Train using NaiveBayes Classifier
def training():
    features_sets = get_features_sets()
    random.shuffle(features_sets)

    train_count = int(len(features_sets) * 0.8)
    train_data = features_sets[:train_count]
    test_data = features_sets[train_count:]

    classifier = NaiveBayesClassifier.train(train_data)
    print(classifier.show_most_informative_features(7))
    print(f"Accuracy = {accuracy(classifier,test_data)}")

    file = open("model.pickle", "wb")
    pickle.dump(classifier, file)
    file.close()

    return classifier

In [13]:
# Antonym and Synonym Function
def get_ant_syn(word):
    syn = set()
    ant = set()

    for syno in wordnet.synsets(word):
        for lemma in syno.lemmas():
            syn.add(lemma.name())
            for anto in lemma.antonyms():
                ant.add(anto.name())

    return list(syn), list(ant)



In [16]:
curr_statement = ""
classifier = None

while True:
    print("1. Tulis Statement")
    print("2. Analisis Statement")
    print("3. Exit")

    choose = input("Pilih")

    if choose == "1":
        statement = input("Tulis statement")

        if len(statement.split()) < 2:
            print("Kurang panjang")
            continue

        curr_statement = statement

        try:
            file = open("model.pickle", "rb")
            classifier = pickle.load(file)
            file.close()
        except:
            print("Training...")
            classifier = training()

    elif choose == "2":
        if not curr_statement:
            print("Gak ada apa2 coba lagi")
            continue

        else: 
            print(f"Curr Sentence = {curr_statement}")

            print("\nPOS_TAG")
            clean_statement = preprocessing(curr_statement)
            pos_tags = pos_tag(clean_statement)

            for text, tag in pos_tags:
                print(f"{text} - {tag}")
            

            print("\nAnt and Syn")
            for word in clean_statement:
                print(f"\n{word}")
                ant,syn = get_ant_syn(word)

                if not ant and not syn:
                    print("Gak ada apa2")
                    continue

                else:
                    print("Syn")
                    if (syn):
                        print(syn)
                    else:
                        print("No Syn")

                    print("Ant")
                    if (ant):
                        print(ant)
                    else:
                        print("No Ant")
            
            print("\nSentiment")
            if classifier:
                print(f"Labeled - {classifier.classify(extract_features(curr_statement))}")
            else:
                print("No Classifier")

    elif choose == "3":
        print("Exiting")
        break


1. Tulis Statement
2. Analisis Statement
3. Exit
Training...
Most Informative Features
                decrease = True           negati : positi =     31.8 : 1.0
                    fell = True           negati : positi =     22.4 : 1.0
                    drop = True           negati : positi =     16.7 : 1.0
                    weak = True           negati : positi =     15.4 : 1.0
                    fall = True           negati : positi =     14.5 : 1.0
                    lose = True           negati : positi =     14.0 : 1.0
                    grow = True           positi : negati =     13.4 : 1.0
None
Accuracy = 0.7753222836095764
1. Tulis Statement
2. Analisis Statement
3. Exit
Curr Sentence = the stocks are going down

POS_TAG
stock - NN
go - VB

Ant and Syn

stock
Syn
No Syn
Ant
['bloodline', 'banal', 'commonplace', 'breed', 'tired', 'neckcloth', 'fund', 'farm_animal', 'stock_up', 'sprout', 'standard', 'blood_line', 'blood', 'parentage', 'inventory', 'stemma', 'pedigree', 's